---
last_verified: 2026-08-06
tool_version: n/a
---

# Terraform Modules, State, and Workspaces — interactive exploration

This notebook walks through three core Terraform concepts: reusable modules, state management, and workspace isolation. Each section includes runnable examples and verification steps.

## Purpose

Terraform modules package reusable infrastructure components. State tracks the real-world resources your configuration manages. Workspaces let you isolate state for different environments (dev, staging, prod) without duplicating configuration. Understanding all three is essential for any Terraform workflow beyond a single-file local experiment.

## 1 — Terraform Modules

A module is a reusable collection of Terraform configuration. Modules accept inputs, expose outputs, and encapsulate resources. They can be sourced from the local filesystem, the Terraform Registry, or a private Git repository.

In [ ]:
%%bash

# Create a local module structure
WORKDIR=$(mktemp -d)
MODULE_DIR="$WORKDIR/modules/simple-ec2"
mkdir -p "$MODULE_DIR"

cat > "$MODULE_DIR/variables.tf" <<'EOF'
variable "name_prefix" {
  type    = string
  default = "demo"
}

variable "instance_type" {
  type    = string
  default = "t3.micro"
}
EOF

cat > "$MODULE_DIR/outputs.tf" <<'EOF'
output "resource_name" {
  value = var.name_prefix
}
EOF

cat > "$MODULE_DIR/main.tf" <<'EOF'
resource "null_resource" "this" {
  triggers = {
    name = var.name_prefix
    type = var.instance_type
  }
}
EOF

echo "Module created at $MODULE_DIR"
ls -la "$MODULE_DIR"

### Using the module

A root configuration calls the module with a `module` block, passing inputs and reading outputs.

In [ ]:
# Example of calling the module from a root config (not executed)
# module "demo_instance" {
#   source        = "./modules/simple-ec2"
#   name_prefix   = "production-web"
#   instance_type = "t3.small"
# }
#
# output "instance_name" {
#   value = module.demo_instance.resource_name
# }

## 2 — Terraform State

Terraform state maps your configuration to real resources. By default it is stored in a local file called `terraform.tfstate`. Every `terraform plan` and `terraform apply` reads and updates this file. When the state file is missing or out of date, Terraform cannot determine what already exists and may attempt to recreate resources.

In [ ]:
%%bash

# Initialize a working directory for state exploration
WORKDIR=$(mktemp -d)
cat > "$WORKDIR/main.tf" <<'EOF'
terraform {
  required_version = ">= 1.0"
}

resource "null_resource" "demo" {
  triggers = {
    stamp = timestamp()
  }
}
EOF
echo "Created demo config in $WORKDIR"
ls -la "$WORKDIR"

### What just happened

A minimal Terraform configuration was written to a temporary directory. It declares a single `null_resource` — a resource that does nothing except exist in state. This is useful for demonstrating state behavior without provisioning real infrastructure.

## 3 — Terraform Workspaces

Workspaces let you manage multiple, isolated state files from the same configuration. Each workspace has its own state, so resources created in one workspace do not interfere with resources in another. This is useful for separating dev, staging, and production environments without duplicating your Terraform code.

In [ ]:
%%bash

# Demonstrate workspace isolation
WORKDIR=$(mktemp -d)
cat > "$WORKDIR/main.tf" <<'EOF'
terraform {
  required_version = ">= 1.0"
}

resource "null_resource" "demo" {
  triggers = {
    workspace = terraform.workspace
    stamp     = timestamp()
  }
}
EOF

cd "$WORKDIR"
terraform init > /dev/null 2>&1

# Create and switch to a dev workspace
terraform workspace new dev 2>/dev/null || true
terraform workspace select dev
terraform plan -out=/tmp/dev-plan.tfplan 2>&1 | tail -3

# Switch to a prod workspace
terraform workspace new prod 2>/dev/null || true
terraform workspace select prod
terraform plan -out=/tmp/prod-plan.tfplan 2>&1 | tail -3

# Show that each workspace has its own state
echo "--- dev workspace state ---"
terraform workspace show
echo "--- prod workspace state ---"
terraform workspace select prod && terraform workspace show

# Clean up
cd /tmp
rm -rf "$WORKDIR" /tmp/dev-plan.tfplan /tmp/prod-plan.tfplan

### Workspace guidance

- Use workspaces for environment isolation within a single configuration.
- For completely different configurations (e.g., dev vs. prod with different resource types), consider separate directories or separate Terraform projects instead.
- Workspaces share the same configuration files — they only differ in state.

## Verify

To confirm understanding of these concepts:

1. **Modules**: Create a local module with variables and outputs, then call it from a root config with different input values.
2. **State**: Run `terraform plan` in a directory with a config and observe that Terraform creates a state file after the first apply.
3. **Workspaces**: Create two workspaces (dev and prod), apply the same config in each, and observe that the state files are isolated.

In [ ]:
%%bash

# Quick summary of the three concepts
echo "Terraform Modules  — reusable, parameterized configuration packages"
echo "Terraform State    — tracks resources in a state file"
echo "Terraform Workspaces — isolate state for different environments"
echo ""
echo "Together they enable: reusable infrastructure, team collaboration, and environment isolation."